## Resource limits

By default a container can consume **all** the host's memory and CPU — so one runaway container can starve everything else. Docker exposes the kernel's **cgroup** limits (module 01's isolation primitive) through a few `run` flags.

**Memory:**

- **`--memory=512m`** — a hard cap. Exceed it and the kernel's **OOM killer** kills the process; the container exits **137** (the exit code from module 01).
- **`--memory-swap=512m`** — the total of memory *plus* swap. Set it equal to `--memory` to disable swap for predictable behaviour.
- **`--memory-reservation=256m`** — a *soft* limit: under pressure the kernel tries to pull the container back toward it, but doesn't kill.

**CPU:**

- **`--cpus=1.5`** — "1.5 CPUs' worth of time" (CFS bandwidth control under the hood). The easy, modern flag.
- **`--cpu-shares=512`** — a *relative weight* (default 1024) that only matters under contention: 512 vs 1024 means half the CPU time when both want it.
- **`--cpuset-cpus="0,1"`** — pin the container to specific cores.

```bash
docker run -d --memory=512m --memory-swap=512m --cpus=1.5 --name api myorg/api
```

**Changing limits on a live container — `docker update`.** You can retune *some* settings without a restart:

```bash
docker update --memory=1g --cpus=2 web
docker update --restart=unless-stopped web
```

`update` changes resource limits and the restart policy in place — the classic use is "this container keeps getting OOM-killed, raise its memory without losing state." But **networking, mounts, env vars, the image, and the entrypoint cannot change on a live container** — for those you must stop, `rm`, and re-create it. That immutability is deliberate: a container's shape is fixed at `run` time.